# CodingBull SEO / GEO evidence audit — 2026-07-26

## TL;DR
The submitted plan needs revision before execution. The GSC Page Indexing export is for **all known pages**, not the submitted sitemap; 21 of 32 non-indexed URLs are redirects or alternative canonicals rather than evidence of 32 broken pages. The query export exposes only 132 of 540 impressions and 2 of 17 clicks, so branded share cannot be inferred. Separate query and page exports also cannot prove which landing page ranked for a query.

This notebook preserves the calculations used in the decision report.

## Context and method

Sources are the two Google Search Console ZIP exports supplied by the site owner. Performance data covers 2026-06-02 through 2026-07-24. Coverage data is current through 2026-07-10. Calculations use each export at its native grain and deliberately do not join Queries.csv to Pages.csv because no shared row-level key exists.

In [1]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import re
import pandas as pd
display = print

PERF_ZIP = Path('/Users/pranshudixit/Downloads/codingbullz.com-Performance-on-Search-2026-07-26.zip')
COVERAGE_ZIP = Path('/Users/pranshudixit/Downloads/codingbullz.com-Coverage-2026-07-26.zip')
REPO = Path('/Users/pranshudixit/code_base/codingbull_immersive_website')

def read_zip_csv(zip_path: Path, member: str) -> pd.DataFrame:
    with ZipFile(zip_path) as archive:
        return pd.read_csv(BytesIO(archive.read(member)))

queries = read_zip_csv(PERF_ZIP, 'Queries.csv')
pages = read_zip_csv(PERF_ZIP, 'Pages.csv')
performance_chart = read_zip_csv(PERF_ZIP, 'Chart.csv')
coverage_chart = read_zip_csv(COVERAGE_ZIP, 'Chart.csv')
coverage_issues = read_zip_csv(COVERAGE_ZIP, 'Critical issues.csv')
coverage_metadata = read_zip_csv(COVERAGE_ZIP, 'Metadata.csv')

print('Performance rows:', len(performance_chart), 'from', performance_chart['Date'].min(), 'to', performance_chart['Date'].max())
display(coverage_metadata)

Performance rows: 53 from 2026-06-02 to 2026-07-24
  Property            Value
0  Sitemap  All known pages


In [2]:
total_clicks = int(performance_chart['Clicks'].sum())
total_impressions = int(performance_chart['Impressions'].sum())
visible_query_clicks = int(queries['Clicks'].sum())
visible_query_impressions = int(queries['Impressions'].sum())

query_completeness = pd.DataFrame([
    {'metric': 'Clicks', 'property_total': total_clicks, 'visible_query_rows': visible_query_clicks, 'omitted_from_query_table': total_clicks-visible_query_clicks},
    {'metric': 'Impressions', 'property_total': total_impressions, 'visible_query_rows': visible_query_impressions, 'omitted_from_query_table': total_impressions-visible_query_impressions},
])
query_completeness['visible_share'] = query_completeness['visible_query_rows'] / query_completeness['property_total']
display(query_completeness)

branded_pattern = re.compile(r'coding\s*bull|codebulls', re.I)
visible_branded = queries[queries['Top queries'].str.contains(branded_pattern, na=False)]
print('Clearly branded rows visible in query table:', int(visible_branded['Impressions'].sum()), 'impressions and', int(visible_branded['Clicks'].sum()), 'clicks')
print('This is a lower bound, not a property-wide brand share.')
display(queries.head(10))

        metric  property_total  ...  omitted_from_query_table  visible_share
0       Clicks              17  ...                        15       0.117647
1  Impressions             540  ...                       411       0.238889

[2 rows x 5 columns]
Clearly branded rows visible in query table: 76 impressions and 1 clicks
This is a lower bound, not a property-wide brand share.
                             Top queries  Clicks  Impressions    CTR  Position
0                            coding bull       1           30  3.33%      5.73
1          software company in ahmedabad       1            2    50%     57.50
2                              codebulls       0           32     0%     17.34
3  "e-commerce inventory and order" saas       0           16     0%      5.50
4                 custom business system       0           11     0%     23.82
5                              techbulls       0            9     0%     51.00
6       coding bull we redesign websites       0            7    

In [3]:
latest_coverage = coverage_chart.dropna(subset=['Indexed', 'Not indexed']).iloc[-1]
indexed = int(latest_coverage['Indexed'])
not_indexed = int(latest_coverage['Not indexed'])
known_urls = indexed + not_indexed
issue_pages = dict(zip(coverage_issues['Reason'], coverage_issues['Pages']))
expected_exclusions = int(issue_pages.get('Page with redirect', 0) + issue_pages.get('Alternative page with proper canonical tag', 0))
needs_diagnosis = not_indexed - expected_exclusions

coverage_summary = pd.DataFrame([
    {'status': 'Indexed', 'pages': indexed, 'interpretation': 'Indexed known URLs'},
    {'status': 'Redirect or alternative canonical', 'pages': expected_exclusions, 'interpretation': 'Usually expected exclusions; verify sitemap hygiene'},
    {'status': 'Needs URL-level diagnosis', 'pages': needs_diagnosis, 'interpretation': '1 redirect error + 3 duplicates + 7 crawled/not indexed'},
])
print('Latest coverage date:', latest_coverage['Date'], '| known URLs:', known_urls)
display(coverage_summary)
display(coverage_issues)

Latest coverage date: 2026-07-10 | known URLs: 51
                              status  ...                                     interpretation
0                            Indexed  ...                                 Indexed known URLs
1  Redirect or alternative canonical  ...  Usually expected exclusions; verify sitemap hy...
2          Needs URL-level diagnosis  ...  1 redirect error + 3 duplicates + 7 crawled/no...

[3 rows x 3 columns]
                                       Reason  ... Pages
0                              Redirect error  ...     1
1                          Page with redirect  ...    18
2   Duplicate without user-selected canonical  ...     3
3  Alternative page with proper canonical tag  ...     3
4             Crawled - currently not indexed  ...     7
5          Discovered – currently not indexed  ...     0

[6 rows x 4 columns]


In [4]:
source_text = (REPO / 'website/src/content/insights.ts').read_text()
article_pattern = re.compile(r"slug:\s*'([^']+)'[\s\S]*?content:\s*`([\s\S]*?)`\s*\.trim\(\),[\s\S]*?date:\s*'([^']+)'")
article_rows = []
for slug, content, date in article_pattern.findall(source_text):
    plain = re.sub(r'[`#*_>\[\]()]', ' ', content)
    words = re.findall(r"\b[\w'-]+\b", plain)
    article_rows.append({'slug': slug, 'published_date': date, 'approx_words': len(words)})
articles = pd.DataFrame(article_rows).sort_values('approx_words')
print('Static insight posts parsed:', len(articles))
print('Posts sharing 2026-05-27:', int((articles['published_date'] == '2026-05-27').sum()))
display(articles)

Static insight posts parsed: 12
Posts sharing 2026-05-27: 8
                                               slug  ... approx_words
5   patient-appointment-booking-system-architecture  ...          596
8              payroll-automation-rules-custom-hrms  ...          613
9       multi-location-attendance-management-system  ...          642
4                clinic-management-software-modules  ...          649
11           saas-vs-custom-software-decision-guide  ...          683
7              shopify-vs-custom-ecommerce-platform  ...          712
6       custom-ecommerce-inventory-order-automation  ...          726
3                  anr-mechanicals-digital-presence  ...          807
10           internal-crm-workflow-automation-guide  ...          942
2           building-hrms-that-scales-500-employees  ...          950
1                  react-nextjs-seo-first-ecommerce  ...         1090
0             why-django-powers-healthcare-backends  ...         1284

[12 rows x 3 columns]


## Decision implications

1. Obtain URL-level examples for the redirect error, duplicates, and crawled-not-indexed groups before changing redirects or requesting indexing.
2. Export query-to-page pairs for the opportunity queries before choosing which URL to expand.
3. Treat raw word count as a diagnostic only. Search intent, verifiable proof, local reputation, useful pricing/timeline information, and authority are the actual decision criteria.
4. Preserve genuine publication dates. Add `dateModified` only after a material edit; do not fabricate staggered dates.